# Technical Documentation RAG System -- Demo

This notebook demonstrates the complete RAG pipeline using the RISC-V technical
documentation corpus. It imports `DemoRunner` from `demo.py` so the logic lives
in one place.

**Prerequisites**: Run `./setup.sh` to install dependencies. The first run
downloads the embedding model (~22 MB) and processes all PDFs (~2 min).
Subsequent runs load from cache.

Three sections:
1. **Pipeline walk-through** -- process PDFs, embed, query
2. **Config comparison** -- same query across retrieval strategies
3. **Agent tools** -- calculator and code analyzer from Epic 5

In [1]:
from demo import DemoRunner, DEMO_QUERIES

runner = DemoRunner(use_cache=True)

# Initialize the system
init = runner.init_system()
print(f"Config:       {init['config']}")
print(f"Architecture: {init['architecture']}")
print(f"Status:       {init['health_status']}")
print(f"Init time:    {init['init_time']:.2f}s")
print()
print("Components:")
for name, ctype in init["components"].items():
    print(f"  {name:25s} {ctype}")

Config:       basic.yaml
Architecture: modular
Status:       healthy
Init time:    5.81s

Components:
  document_processor        ModularDocumentProcessor
  embedder                  ModularEmbedder
  retriever                 ModularUnifiedRetriever
  answer_generator          AnswerGenerator
  query_processor           ModularQueryProcessor


## 1. Pipeline Walk-Through

The pipeline processes PDF files through four stages:
1. **Document processing** -- extract text, split into chunks
2. **Embedding** -- encode chunks with sentence-transformers (384-dim vectors)
3. **Indexing** -- store in FAISS for dense search + BM25 for sparse search
4. **Query** -- retrieve relevant chunks, fuse scores, generate answer

Answer generation uses a mock LLM (templated responses). Retrieval is real.

In [2]:
# Process the RISC-V corpus (cached after first run)
corpus = runner.process_corpus()

if corpus.get("from_cache"):
    print(f"Loaded from cache: {corpus['total_chunks']} chunks from {corpus['total_docs']} files")
else:
    print(f"Processed {corpus['total_docs']} files -> {corpus['total_chunks']} chunks in {corpus['elapsed']:.1f}s")
    if corpus.get("failed"):
        print(f"Failed: {corpus['failed']} files")

Loaded from cache: 4395 chunks from 73 files


In [3]:
# Query the system with three curated questions
for q in DEMO_QUERIES:
    result = runner.query(q)
    print(f"Q: {result['query']}")
    print(f"Confidence: {result['confidence']:.2f}  |  Time: {result['timing']:.2f}s")
    print(f"Answer: {result['answer'][:300]}")
    if result["sources"]:
        print(f"Top source: {result['sources'][0]['file']}")
        if result["scores"]:
            print(f"Top score:  {result['scores'][0]:.4f}")
    print("-" * 70)

Q: What is the RISC-V vector extension and what operations does it support?
Confidence: 0.80  |  Time: 0.17s
Answer: Based on the technical documentation, RISC-V is implemented using industry best practices. The implementation details show that it uses advanced algorithms for efficient processing [Document 5].
Top source: data/riscv_corpus/research/papers/performance-analysis/2507.01457v1.pdf
Top score:  0.0258
----------------------------------------------------------------------


Q: How does RISC-V handle privilege levels and memory protection?
Confidence: 0.73  |  Time: 0.15s
Answer: Based on the technical documentation, RISC-V is implemented using industry best practices. The implementation details show that performance benchmarks show significant improvements [Document 5].
Top source: data/riscv_corpus/research/papers/security-studies/2107.04175v1.pdf
Top score:  0.0258
----------------------------------------------------------------------


Q: What are the security challenges in RISC-V implementations?
Confidence: 0.73  |  Time: 0.14s
Answer: Based on the technical documentation, RISC-V is a key component of the system architecture. The implementation details show that performance benchmarks show significant improvements [Document 5].
Top source: data/riscv_corpus/research/papers/security-studies/2107.04175v1.pdf
Top score:  0.0311
----------------------------------------------------------------------


## 2. Config Comparison

The system is config-driven: same codebase, different YAML, different behavior.
Here we run one query through three configurations to compare retrieval strategies:

| Config | Fusion | Reranker |
|--------|--------|----------|
| `basic.yaml` | Reciprocal Rank Fusion | Identity (pass-through) |
| `epic2_graph_enhanced_mock.yaml` | Graph-enhanced RRF | Neural cross-encoder |
| `epic2_score_aware_mock.yaml` | Score-aware fusion | Neural cross-encoder |

Documents are embedded once and reused across all three configs.

In [4]:
q = DEMO_QUERIES[0]
comparison = runner.compare_configs(q)

print(f"Query: {q}")
print()
print(f"{'Config':40s} {'Fusion':20s} {'Reranker':10s} {'Conf':>6s} {'Top Score':>10s} {'Time':>7s}")
print(f"{'-'*40} {'-'*20} {'-'*10} {'-'*6} {'-'*10} {'-'*7}")
for r in comparison:
    if r.get("error"):
        print(f"{r['config']:40s} {r['fusion']:20s} {r['reranker']:10s} ERROR")
        print(f"  {r['error'][:80]}")
    else:
        print(f"{r['config']:40s} {r['fusion']:20s} {r['reranker']:10s} "
              f"{r['confidence']:6.2f} {r['top_score']:10.4f} {r['timing']:6.1f}s")

Query: What is the RISC-V vector extension and what operations does it support?

Config                                   Fusion               Reranker     Conf  Top Score    Time
---------------------------------------- -------------------- ---------- ------ ---------- -------
basic.yaml                               RRF                  identity     0.76     0.0258    0.9s
epic2_graph_enhanced_mock.yaml           graph-enhanced RRF   neural       0.78     1.0000   15.7s
epic2_score_aware_mock.yaml              score-aware          neural       0.82     1.0000    1.6s


## 3. Epic 5 Agent Tools

The Epic 5 agent system includes a tool registry with typed parameters.
Tools are standalone components that can be used by the ReAct agent or
called directly. Here we exercise the calculator and code analyzer.

In [5]:
tools = runner.demo_tools()

# Calculator results
print("Calculator:")
for r in tools["calculator_results"]:
    if r["success"]:
        print(f"  {r['expression']:35s} = {r['result']}")
    else:
        print(f"  {r['expression']:35s} -> ERROR: {r['error']}")

# Code analyzer
print()
print("Code Analyzer (RISC-V instruction decoder):")
ar = tools["analyzer_results"]
if ar["success"]:
    print(ar["analysis"])

# Registry stats
print()
stats = tools["stats"]
print(f"Registry: {stats['total_tools']} tools, {stats['total_executions']} executions, "
      f"{stats['overall_success_rate']:.0%} success rate")

/Users/apa/ml_projects/migration-workspace/rag-portfolio-extract/src/components/query_processors/agents/langchain_adapter.py:52: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class PhaseOneToolAdapter(LangChainBaseTool):
calculator execution failed: Division by zero


Calculator:
  2 ** 10                             = 1024
  sqrt(144) + log(e)                  = 13
  sin(pi / 4) * cos(pi / 4)           = 0.5
  1 / 0                               -> ERROR: Division by zero

Code Analyzer (RISC-V instruction decoder):
Code Analysis: SUCCESS

=== Basic Metrics ===
Syntax Valid: Yes
Lines of Code: 42
Total Statements: 30
Functions: 2
Classes: 0
Imports: 0
Has Main Block: No

=== Functions ===
  decode_instruction(raw) [has docstring]
    Statements: 9
  _classify_format(opcode) [has docstring]
    Statements: 21

=== Code Quality Notes ===
  ✓ Code includes documentation (docstrings)
  Average function complexity: 15.0 statements

Registry: 2 tools, 5 executions, 80% success rate


## Architecture Summary

```
config/*.yaml
     |
     v
PlatformOrchestrator
  |-- DocumentProcessor  (hybrid_pdf -> chunks)
  |-- Embedder           (sentence-transformers -> 384-dim vectors)
  |-- Retriever          (FAISS + BM25 -> fusion -> reranker)
  |-- AnswerGenerator    (mock / ollama / openai / ...)
  +-- QueryProcessor     (Epic 5 agent with tools)
```

Everything is assembled from config at startup. Swap the YAML, get a different
pipeline. The demo above shows three configs producing measurably different
retrieval behavior from the same codebase and corpus.